In [ ]:
import warnings
import os

# Suppression spécifique du warning pkg_resources
warnings.filterwarnings("ignore", category=UserWarning, module="face_recognition_models")
# Suppression également des autres warnings FutureWarning
warnings.filterwarnings("ignore", category=FutureWarning)

import cv2
import numpy as np
import face_recognition
from ultralytics import YOLO
from collections import defaultdict
from datetime import datetime
import time

# ===========================
# 🔊 MEMBRE 3 - ALERTES VOCALES (CORRIGÉ)
# ===========================
import pyttsx3
import threading

# Initialisation du moteur TTS (une seule fois au démarrage)
_tts_engine = None

def _init_tts():
    global _tts_engine
    try:
        _tts_engine = pyttsx3.init()
        _tts_engine.setProperty('rate', 160)
        _tts_engine.setProperty('volume', 1.0)
        
        # Chercher une voix française
        voices = _tts_engine.getProperty('voices')
        for voice in voices:
            if 'fr' in voice.id.lower() or 'french' in voice.name.lower():
                _tts_engine.setProperty('voice', voice.id)
                print(f"✅ [MEMBRE 3] Voix: {voice.name}")
                break
        
        print("✅ [MEMBRE 3] Synthèse vocale prête!")
        return True
    except Exception as e:
        print(f"⚠️ [MEMBRE 3] Erreur TTS: {e}")
        return False

def dire(message):
    """Joue un message vocal sans bloquer le programme principal"""
    print(f"🔊 [MEMBRE 3] {message}")
    
    def _jouer():
        if _tts_engine:
            try:
                _tts_engine.say(message)
                _tts_engine.runAndWait()
            except Exception as e:
                print(f"⚠️ Erreur audio: {e}")
        else:
            print('\a')  # Bip système
    
    # Démarrer dans un thread séparé
    threading.Thread(target=_jouer, daemon=True).start()

# Démarrer le TTS
_init_tts()

# ── Variables globales Membre 3 ──────────────────────────
_alerte_limite_jouee   = False
_derniere_alerte_temps = 0.0
LIMITE_PERSONNES       = 1

def membre3_alerte_personne_trouvee(nom, confiance):
    """Appeler quand la personne recherchée est détectée (caméra mode 2)"""
    global _derniere_alerte_temps
    maintenant = time.time()
    if maintenant - _derniere_alerte_temps >= 5:
        _derniere_alerte_temps = maintenant
        dire(f"{nom} détectée, confiance {confiance:.0%}")

def membre3_alerte_nouveau_comptage(compteur, nom=""):
    global _alerte_limite_jouee
    
    if compteur > LIMITE_PERSONNES and not _alerte_limite_jouee:
        _alerte_limite_jouee = True
        msg_limite = f"Attention ! Vous avez dépassé la limite de {LIMITE_PERSONNES} personnes !"
        print(f"🔊 [MEMBRE 3] {msg_limite}")
        dire(msg_limite)

def membre3_reset():
    global _alerte_limite_jouee, _derniere_alerte_temps
    _alerte_limite_jouee   = False
    _derniere_alerte_temps = 0.0
    dire("Compteur remis à zéro.")

# ===========================
# 1. CHARGER LES IMAGES DE RÉFÉRENCE
# ===========================
def charger_references_face_recognition():
    known_encodings = []
    known_names = []
    
    dossier = "personnes_reference"
    
    if not os.path.exists(dossier):
        print(f"❌ Dossier '{dossier}' introuvable !")
        os.makedirs(dossier)
        print(f"✅ Dossier '{dossier}' créé. Ajoutez-y vos photos !")
        return known_encodings, known_names
    
    print(f"📁 Chargement des références depuis '{dossier}':")
    
    for fichier in os.listdir(dossier):
        if fichier.lower().endswith(('.jpg', '.jpeg', '.png')):
            nom = os.path.splitext(fichier)[0]
            chemin = os.path.join(dossier, fichier)
            
            try:
                image = face_recognition.load_image_file(chemin)
                encodings = face_recognition.face_encodings(image)
                
                if encodings:
                    known_encodings.append(encodings[0])
                    known_names.append(nom)
                    print(f"  ✅ {nom} - Visage encodé avec succès")
                else:
                    print(f"  ⚠️ {fichier} - Aucun visage détecté")
            except Exception as e:
                print(f"  ❌ Erreur avec {fichier}: {e}")
    
    print(f"\n📊 Total: {len(known_names)} personne(s) de référence chargée(s)")
    return known_encodings, known_names

# ===========================
# 2. SÉLECTIONNER FICHIER AVEC EXPLORATEUR
# ===========================
def selectionner_image_via_dialogue():
    try:
        from tkinter import Tk, filedialog
        root = Tk()
        root.attributes('-topmost', True)
        root.update()
        chemin = filedialog.askopenfilename(
            title="Sélectionner une image",
            filetypes=[("Images", "*.jpg *.jpeg *.png *.bmp *.gif"), ("Tous les fichiers", "*.*")]
        )
        root.destroy()
        return chemin if chemin else None
    except:
        print("\n📁 Entrez le chemin de l'image:")
        chemin = input("👉 ").strip().strip('\'" ')
        return chemin if chemin else None

def selectionner_video_via_dialogue():
    try:
        from tkinter import Tk, filedialog
        root = Tk()
        root.attributes('-topmost', True)
        root.update()
        chemin = filedialog.askopenfilename(
            title="Sélectionner une vidéo",
            filetypes=[("Vidéos", "*.mp4 *.avi *.mov *.mkv *.flv *.wmv"), ("Tous les fichiers", "*.*")]
        )
        root.destroy()
        return chemin if chemin else None
    except:
        print("\n📁 Entrez le chemin de la vidéo:")
        chemin = input("👉 ").strip().strip('\'" ')
        return chemin if chemin else None

# ===========================
# 3. ANALYSE D'UNE IMAGE
# ===========================
def analyser_image():
    print("\n" + "="*50)
    print("🖼️  ANALYSE D'UNE IMAGE")
    print("="*50)
    
    known_encodings, known_names = charger_references_face_recognition()
    
    if not known_encodings:
        print("\n❌ AUCUNE RÉFÉRENCE CHARGÉE!")
        print("💡 Créez un dossier 'personnes_reference' avec des photos")
        return
    
    print("\n📸 Sélectionnez l'image à analyser...")
    chemin_image = selectionner_image_via_dialogue()
    
    if not chemin_image or not os.path.exists(chemin_image):
        print("❌ Image non trouvée!")
        return
    
    print(f"✅ Image sélectionnée: {os.path.basename(chemin_image)}")
    
    image = cv2.imread(chemin_image)
    if image is None:
        print("❌ Impossible de charger l'image")
        return
    
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    print("\n🔍 Analyse en cours...")
    face_locations = face_recognition.face_locations(rgb_image)
    face_encodings = face_recognition.face_encodings(rgb_image, face_locations)
    
    print(f"✅ {len(face_locations)} visage(s) détecté(s)\n")
    
    compteur_reconnus = 0
    compteur_inconnus = 0
    stats_personnes = defaultdict(int)
    
    for i, (face_encoding, (top, right, bottom, left)) in enumerate(zip(face_encodings, face_locations)):
        matches = face_recognition.compare_faces(known_encodings, face_encoding, tolerance=0.5)
        name = "Inconnu"
        couleur = (0, 0, 255)
        
        if any(matches):
            face_distances = face_recognition.face_distance(known_encodings, face_encoding)
            best_match_index = np.argmin(face_distances)
            if matches[best_match_index]:
                name = known_names[best_match_index]
                confidence = 1 - face_distances[best_match_index]
                couleur = (0, 255, 0)
                compteur_reconnus += 1
                stats_personnes[name] += 1
                print(f"👤 Visage {i+1}: {name} (confiance: {confidence:.2%})")
            else:
                compteur_inconnus += 1
                print(f"❓ Visage {i+1}: Inconnu")
        else:
            compteur_inconnus += 1
            print(f"❓ Visage {i+1}: Inconnu")
        
        cv2.rectangle(image, (left, top), (right, bottom), couleur, 2)
        cv2.putText(image, name, (left, top-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, couleur, 2)
    
    print("\n" + "="*50)
    print("📊 STATISTIQUES DE L'IMAGE")
    print("="*50)
    print(f"Total visages: {len(face_locations)}")
    print(f"Visages reconnus: {compteur_reconnus}")
    print(f"Visages inconnus: {compteur_inconnus}")
    
    if stats_personnes:
        print("\n👥 Personnes identifiées:")
        for personne, count in stats_personnes.items():
            print(f"  - {personne}: {count} fois")
    
    nom_sortie = f"resultat_{os.path.basename(chemin_image)}"
    cv2.imwrite(nom_sortie, image)
    print(f"\n💾 Résultat sauvegardé: {nom_sortie}")
    
    cv2.imshow("Analyse d'image", image)
    print("\nℹ️ Appuyez sur une touche pour fermer l'image...")
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# ===========================
# 4. RECHERCHE D'UNE PERSONNE EN MODE CAMÉRA
# ===========================
def rechercher_personne_camera_avec_image():
    print("\n" + "="*50)
    print("🔍 RECHERCHE D'UNE PERSONNE AVEC VOTRE IMAGE")
    print("="*50)
    
    print("\n📸 Sélectionnez l'image de la personne à rechercher...")
    chemin_image_recherche = selectionner_image_via_dialogue()
    
    if not chemin_image_recherche or not os.path.exists(chemin_image_recherche):
        print("❌ Image non trouvée!")
        return
    
    print(f"\n✅ Image chargée: {os.path.basename(chemin_image_recherche)}")
    
    print("\n⏳ Analyse de l'image fournie...")
    image_recherche = face_recognition.load_image_file(chemin_image_recherche)
    encodings_recherche = face_recognition.face_encodings(image_recherche)
    
    if not encodings_recherche:
        print("❌ Aucun visage détecté dans l'image fournie!")
        dire("Aucun visage détecté dans l'image fournie")
        return
    
    encodage_cible = encodings_recherche[0]
    
    personne_cible = input("\n🏷️  Entrez un nom pour cette personne: ")
    if not personne_cible:
        personne_cible = os.path.splitext(os.path.basename(chemin_image_recherche))[0]
    
    print(f"\n✅ Recherche de '{personne_cible}' activée!")
    print("📷 Ouverture de la caméra...")
    
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Impossible d'ouvrir la caméra !")
        return
    
    SEUIL_RECONNAISSANCE = 0.5
    TAUX_REDUCTION = 0.5
    
    detection_active = False
    temps_derniere_detection = 0
    alerte_sonore = True
    
    print("\n🚀 Recherche en cours...")
    print("ℹ️ Appuyez sur 'q' pour quitter | 's' pour son ON/OFF")
    print(f"🎯 Personne recherchée: {personne_cible}")
    
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        
        small_frame = cv2.resize(frame, (0, 0), fx=TAUX_REDUCTION, fy=TAUX_REDUCTION)
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
        
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations, num_jitters=0)
        personne_trouvee = False
        
        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            match = face_recognition.compare_faces([encodage_cible], face_encoding, tolerance=SEUIL_RECONNAISSANCE)
            
            top = int(top / TAUX_REDUCTION)
            right = int(right / TAUX_REDUCTION)
            bottom = int(bottom / TAUX_REDUCTION)
            left = int(left / TAUX_REDUCTION)
            
            if match[0]:
                personne_trouvee = True
                detection_active = True
                temps_derniere_detection = frame_count
                
                distances = face_recognition.face_distance([encodage_cible], face_encoding)
                confiance = 1 - distances[0]
                
                cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 3)
                cv2.putText(frame, f"{personne_cible} ({confiance:.1%})", (left, top-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                cv2.putText(frame, "!!! PERSONNE DETECTEE !!!", (50, 100), 
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
                
                if alerte_sonore:
                    print(f"\n🔔 ALERTE! {personne_cible} détecté(e) (confiance: {confiance:.1%})")
                    membre3_alerte_personne_trouvee(personne_cible, confiance)
            else:
                cv2.rectangle(frame, (left, top), (right, bottom), (0, 0, 255), 2)
                cv2.putText(frame, "Autre personne", (left, top-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        
        if frame_count - temps_derniere_detection > 30:
            detection_active = False
            if not detection_active and not personne_trouvee and frame_count % 150 == 0:
                dire(f"Aucune personne détectée. Je recherche {personne_cible}.")
        
        status = "ACTIVE" if detection_active else "RECHERCHE EN COURS"
        couleur_status = (0, 255, 0) if detection_active else (0, 0, 255)
        
        cv2.rectangle(frame, (10, 10), (450, 150), (0, 0, 0), -1)
        cv2.putText(frame, f"RECHERCHE: {personne_cible}", (20, 40), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Statut: {status}", (20, 70), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, couleur_status, 2)
        cv2.putText(frame, f"Son: {'ON' if alerte_sonore else 'OFF'}", (20, 100), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
        
        cv2.imshow('Recherche de personne - Camera', frame)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('s'):
            alerte_sonore = not alerte_sonore
            print(f"🔊 Son: {'ACTIVE' if alerte_sonore else 'DESACTIVE'}")
    
    cap.release()
    cv2.destroyAllWindows()

# ===========================
# 5. RECHERCHER UNE PERSONNE DANS UNE VIDÉO (VERSION CORRIGÉE)
# ===========================
def rechercher_personne_dans_video():
    """Recherche une personne spécifique dans une vidéo - ALERTE VOCALE SIMPLE"""
    print("\n" + "="*50)
    print("🎬 RECHERCHE D'UNE PERSONNE DANS UNE VIDÉO")
    print("="*50)
    
    print("\n📸 Étape 1/2: Sélectionnez l'image de la personne à rechercher...")
    chemin_image_recherche = selectionner_image_via_dialogue()
    
    if not chemin_image_recherche or not os.path.exists(chemin_image_recherche):
        print("❌ Image non trouvée!")
        return
    
    print(f"✅ Image chargée: {os.path.basename(chemin_image_recherche)}")
    
    print("\n⏳ Analyse de l'image...")
    image_recherche = face_recognition.load_image_file(chemin_image_recherche)
    encodings_recherche = face_recognition.face_encodings(image_recherche)
    
    if not encodings_recherche:
        print("❌ Aucun visage détecté dans l'image!")
        dire("Aucun visage détecté dans l'image fournie")
        return
    
    encodage_cible = encodings_recherche[0]
    
    personne_cible = input("\n🏷️  Nom de la personne: ")
    if not personne_cible:
        personne_cible = os.path.splitext(os.path.basename(chemin_image_recherche))[0]
    
    print(f"\n🎬 Étape 2/2: Sélectionnez la vidéo...")
    chemin_video = selectionner_video_via_dialogue()
    
    if not chemin_video or not os.path.exists(chemin_video):
        print("❌ Vidéo non trouvée!")
        return
    
    print(f"✅ Vidéo sélectionnée: {os.path.basename(chemin_video)}")
    
    cap = cv2.VideoCapture(chemin_video)
    if not cap.isOpened():
        print("❌ Impossible d'ouvrir la vidéo!")
        return
    
    SEUIL_RECONNAISSANCE = 0.5
    TAUX_REDUCTION = 0.5
    FPS = int(cap.get(cv2.CAP_PROP_FPS))
    frame_count = 0
    extraction_interval = max(1, FPS // 4)
    
    personne_trouvee = False
    dossier_captures = f"captures_{personne_cible}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(dossier_captures, exist_ok=True)
    
    print(f"\n🔍 Recherche de '{personne_cible}' dans la vidéo...")
    print("🎬 Appuyez sur 'q' pour quitter | 's' pour sauvegarder | ESPACE pour pause")
    
    pause = False
    display_frame = None
    alerte_donnee = False
    
    while True:
        if not pause:
            ret, frame = cap.read()
            if not ret:
                print("\n🏁 Fin de la vidéo atteinte!")
                break
            
            frame_count += 1
            
            if frame_count % extraction_interval == 0:
                small_frame = cv2.resize(frame, (0, 0), fx=TAUX_REDUCTION, fy=TAUX_REDUCTION)
                rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
                
                face_locations = face_recognition.face_locations(rgb_small_frame)
                face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)
                
                temps_actuel = frame_count / FPS
                
                for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
                    match = face_recognition.compare_faces([encodage_cible], face_encoding, tolerance=SEUIL_RECONNAISSANCE)
                    
                    if match[0]:
                        if not personne_trouvee:
                            personne_trouvee = True
                            minutes = int(temps_actuel // 60)
                            secondes = int(temps_actuel % 60)
                            print(f"\n✅ {personne_cible} TROUVÉ(E) à {minutes:02d}:{secondes:02d} !")
                            
                            if not alerte_donnee:
                                dire(f"{personne_cible} trouvée dans la vidéo")
                                alerte_donnee = True
                        
                        top = int(top / TAUX_REDUCTION)
                        right = int(right / TAUX_REDUCTION)
                        bottom = int(bottom / TAUX_REDUCTION)
                        left = int(left / TAUX_REDUCTION)
                        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 3)
                        cv2.putText(frame, f"{personne_cible} (TROUVEE!)", (left, top-10), 
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                        
                        if len(os.listdir(dossier_captures)) == 0:
                            nom_capture = f"{dossier_captures}/{personne_cible}_premiere_detection.jpg"
                            cv2.imwrite(nom_capture, frame)
                            print(f"   💾 Capture: {os.path.basename(nom_capture)}")
        
        display_frame = frame.copy() if not pause and 'frame' in locals() else display_frame
        
        if display_frame is not None:
            temps_actuel = frame_count / FPS if not pause else temps_actuel
            minutes_actu = int(temps_actuel // 60)
            secondes_actu = int(temps_actuel % 60)
            
            cv2.rectangle(display_frame, (10, 10), (550, 150), (0, 0, 0), -1)
            cv2.putText(display_frame, f"RECHERCHE: {personne_cible}", (20, 40), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(display_frame, f"Temps: {minutes_actu:02d}:{secondes_actu:02d}", (20, 70), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)
            
            if personne_trouvee:
                cv2.putText(display_frame, ">>> PERSONNE TROUVEE ! <<<", (20, 100), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            else:
                cv2.putText(display_frame, ">>> RECHERCHE EN COURS... <<<", (20, 100), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            
            cv2.putText(display_frame, f"Statut: {'PAUSE' if pause else 'LECTURE'}", (20, 130), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255) if pause else (0, 255, 0), 1)
            
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames > 0:
                progression = (frame_count / total_frames) * 100
                barre_width = int(display_frame.shape[1] * progression / 100)
                cv2.rectangle(display_frame, (0, display_frame.shape[0]-5), 
                             (barre_width, display_frame.shape[0]), (0, 255, 0), -1)
            
            cv2.imshow('Recherche dans Video - Q pour quitter', display_frame)
        
        key = cv2.waitKey(30) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('s') and 'frame' in locals():
            timestamp = datetime.now().strftime("%H%M%S")
            nom_manuel = f"{dossier_captures}/manuel_{personne_cible}_{timestamp}.jpg"
            cv2.imwrite(nom_manuel, frame)
            print(f"💾 Capture manuelle: {os.path.basename(nom_manuel)}")
        elif key == ord(' '):
            pause = not pause
            print(f"⏸️  {'Pause' if pause else 'Reprise'}")
    
    cap.release()
    cv2.destroyAllWindows()
    
    print("\n" + "="*50)
    print("📊 RAPPORT DE RECHERCHE")
    print("="*50)
    print(f"👤 Personne recherchée: {personne_cible}")
    print(f"🎬 Vidéo: {os.path.basename(chemin_video)}")
    
    if personne_trouvee:
        print(f"✅ RÉSULTAT: {personne_cible} a été trouvé(e) dans la vidéo !")
        if not alerte_donnee:
            dire(f"{personne_cible} trouvée dans la vidéo")
    else:
        print(f"❌ RÉSULTAT: {personne_cible} n'a PAS été trouvé(e) dans la vidéo")
        dire(f"{personne_cible} non trouvée dans la vidéo")
    
    print("="*50)

# ===========================
# 6. MODE CAMERA AVEC FACE_RECOGNITION ET COMPTAGE
# ===========================
def mode_camera_face_recognition():
    print("=" * 50)
    print("🔍 DÉMARRAGE DE LA RECONNAISSANCE FACIALE")
    print("=" * 50)
    
    print("⏳ Chargement YOLO...")
    model = YOLO("yolov8n.pt")
    print("✅ YOLO prêt!")
    
    print("\n⏳ Chargement des visages de référence...")
    known_encodings, known_names = charger_references_face_recognition()
    
    if not known_encodings:
        print("\n❌ AUCUNE RÉFÉRENCE CHARGÉE!")
        return
    
    print("\n📷 Ouverture de la caméra...")
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("❌ Impossible d'ouvrir la caméra !")
        return
    
    SEUIL_RECONNAISSANCE = 0.5
    TAUX_REDUCTION = 0.5
    
    personnes_deja_comptees = set()
    compteur_total = 0
    derniere_frame_par_personne = {}
    FRAME_TIMEOUT = 60
    
    temps_debut = datetime.now()
    frame_count = 0
    
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"✅ Caméra ouverte: {frame_width}x{frame_height}")
    print("ℹ️ Appuyez sur 'q' pour quitter | 'r' pour réinitialiser\n")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        
        results = model.predict(frame, verbose=False, imgsz=320)
        personnes_yolo = 0
        
        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                if cls == 0 and float(box.conf[0]) > 0.5:
                    personnes_yolo += 1
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 128, 0), 1)
        
        small_frame = cv2.resize(frame, (0, 0), fx=TAUX_REDUCTION, fy=TAUX_REDUCTION)
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
        
        face_locations = face_recognition.face_locations(rgb_small_frame)
        
        if face_locations:
            face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)
            
            for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
                matches = face_recognition.compare_faces(known_encodings, face_encoding, tolerance=SEUIL_RECONNAISSANCE)
                
                name = "Inconnu"
                couleur = (0, 0, 255)
                
                if any(matches):
                    face_distances = face_recognition.face_distance(known_encodings, face_encoding)
                    best_match_index = np.argmin(face_distances)
                    if matches[best_match_index]:
                        name = known_names[best_match_index]
                        couleur = (0, 255, 0)
                        
                        if name not in personnes_deja_comptees:
                            personnes_deja_comptees.add(name)
                            compteur_total += 1
                            print(f"✅ NOUVELLE PERSONNE ! {name} (Total: {compteur_total})")
                            membre3_alerte_nouveau_comptage(compteur_total, nom=name)
                        
                        derniere_frame_par_personne[name] = frame_count
                else:
                    face_hash = hash(tuple(np.round(face_encoding[:10], 3)))
                    inconnu_id = f"inconnu_{face_hash}"
                    
                    if inconnu_id not in personnes_deja_comptees:
                        personnes_deja_comptees.add(inconnu_id)
                        compteur_total += 1
                        print(f"❓ NOUVEL INCONNU ! (Total: {compteur_total})")
                        membre3_alerte_nouveau_comptage(compteur_total)
                    
                    derniere_frame_par_personne[inconnu_id] = frame_count
                
                top = int(top / TAUX_REDUCTION)
                right = int(right / TAUX_REDUCTION)
                bottom = int(bottom / TAUX_REDUCTION)
                left = int(left / TAUX_REDUCTION)
                
                cv2.rectangle(frame, (left, top), (right, bottom), couleur, 2)
                cv2.putText(frame, name, (left, top-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, couleur, 2)
        else:
            if frame_count % 200 == 0:
                dire("Aucune personne détectée.")
        
        to_remove = []
        for person, last_frame in derniere_frame_par_personne.items():
            if frame_count - last_frame > FRAME_TIMEOUT:
                to_remove.append(person)
        for person in to_remove:
            del derniere_frame_par_personne[person]
        
        temps_ecoule = (datetime.now() - temps_debut).seconds
        
        overlay = frame.copy()
        cv2.rectangle(overlay, (frame_width - 300, 10), (frame_width - 10, 150), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
        
        y_offset = 30
        cv2.putText(frame, "STATISTIQUES:", (frame_width - 290, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)
        cv2.putText(frame, f"PERSONNES UNIQUES: {compteur_total}", (frame_width - 290, y_offset + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.putText(frame, f"Visages: {len(face_locations)}", (frame_width - 290, y_offset + 55), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
        cv2.putText(frame, f"YOLO: {personnes_yolo}", (frame_width - 290, y_offset + 75), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
        cv2.putText(frame, f"Refs: {len(known_names)} | Duree: {temps_ecoule}s", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, "Press 'q' quit | 'r' reset", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        cv2.imshow('Reconnaissance Faciale', frame)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('r'):
            print("\n🔄 RÉINITIALISATION...")
            personnes_deja_comptees.clear()
            derniere_frame_par_personne.clear()
            compteur_total = 0
            temps_debut = datetime.now()
            print("✅ Compteur remis à zéro!")
            membre3_reset()
    
    cap.release()
    cv2.destroyAllWindows()
    
    temps_total = (datetime.now() - temps_debut).seconds
    print("\n" + "="*50)
    print("📊 RAPPORT FINAL")
    print("="*50)
    print(f"⏱️  Durée: {temps_total} secondes")
    print(f"👥 PERSONNES UNIQUES VUES: {compteur_total}")
    print("="*50)
    print("👋 Au revoir!")

# ===========================
# 7. LANCEMENT PRINCIPAL
# ===========================
if __name__ == "__main__":
    print("\n" + "="*50)
    print("🎯 RECONNAISSANCE FACIALE")
    print("="*50)
    print("\nChoisissez une option:")
    print("1. Mode Caméra (reconnaissance en temps réel + comptage)")
    print("2. Rechercher une personne avec MA PROPRE IMAGE (caméra)")
    print("3. Analyser une image statique")
    print("4. Rechercher une personne dans une vidéo")
    print("5. Quitter")
    
    choix = input("\nVotre choix (1-5): ")
    
    if choix == "1":
        mode_camera_face_recognition()
    elif choix == "2":
        rechercher_personne_camera_avec_image()
    elif choix == "3":
        analyser_image()
    elif choix == "4":
        rechercher_personne_dans_video()
    else:
        print("👋 Au revoir!")